# TrustSR Phase 0 — real WorldStrat fine-tuning smoke test

**Not executed in Colab yet.** This notebook does not train from scratch and cannot pass without genuine prepared WorldStrat pairs. Select **Runtime → Change runtime type → GPU**. See `docs/r5_data_contract.md` for the required paired-data provenance and grid preparation.

Export the project on your machine with `git archive --format=zip --output=trustsr-phase0.zip HEAD`. That includes tracked code/configs, not ignored datasets. Upload the project archive below. The next upload accepts a prepared-data archive containing `data/worldstrat/pairs.json` and its relative COG files. Do not upload the full WorldStrat collection.


In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile

def unpack_uploaded_zip(destination):
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one project or prepared-data ZIP')
    name = next(iter(uploaded))
    archive_path = Path(name).resolve()
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Unsafe archive path or symlink')
        archive.extractall(destination)
    return destination

project = unpack_uploaded_zip(Path.cwd() / 'TrustSR')
os.chdir(project)
assert Path('configs/phase0.yaml').is_file(), 'Project ZIP must contain configs at its root'


## Install pinned dependencies

These direct pins were tested on native Windows/Python 3.13 with CUDA 12.6. Colab is a separate environment: this installation must succeed there, and its package list is recorded below. Restart the runtime if Colab requests it, then return to the project directory. Do not silently substitute CPU PyTorch.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-phase0.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


## Upload prepared real pairs

Use the small prepared-data archive described above. Its manifest must list at least three actual training AOIs with named RGBN COGs, hashes, source URLs and correct scale/offsets. The notebook will not create sample imagery or targets.


In [ ]:
unpack_uploaded_zip(project)
assert Path('data/worldstrat/pairs.json').is_file(), 'Missing prepared real-data manifest'


In [ ]:
import torch, platform, json
from pathlib import Path
assert torch.cuda.is_available(), 'Select a Colab GPU runtime; CPU is not an R5 pass'
print(platform.platform(), torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
Path('risk/results').mkdir(parents=True, exist_ok=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
Path('risk/results/colab-packages.txt').write_text(freeze)
result = subprocess.run([sys.executable, '-m', 'risk.r5_finetune', '--config', 'configs/phase0.yaml'])
report = json.loads(Path('risk/results/r5.json').read_text())
print(json.dumps(report, indent=2))
assert result.returncode == 0 and report['status'] == 'PASS', 'R5 failed or is blocked; preserve the report'


## Download evidence

The saved result must include 50 real losses, weight changes, checkpoint hash and reload agreement. It is a smoke test only. Do not proceed to Phase 1 from this notebook.


In [ ]:
import yaml
cfg = yaml.safe_load(Path('configs/phase0.yaml').read_text())
files.download('risk/results/r5.json')
files.download('risk/results/colab-packages.txt')
checkpoint = Path(cfg['training']['checkpoint'])
if checkpoint.is_file():
    files.download(str(checkpoint))
